# OPERA paper figures — length generalization and decoding cost

Evaluation only: **no training**. Loads the finished runs on Drive (the FineWeb-Edu ladder from
`OPERA_Scale`, the 172M run, the byte-transformer baselines) and produces two paper figures:

1. **Per-position loss past the training length** (`experiments/eval_position.py`): OPERA vs the
   byte transformer on the same held-out FineWeb documents, out to 4× the 1,024-byte training length.
2. **Decoding cost vs context** (`experiments/bench_decode.py`): decoder state and time per byte,
   OPERA's Fenwick decoder vs a transformer with a KV cache, at matched width, up to 64k bytes.

Everything is written next to the runs on Drive (`runs/position_curve_fineweb.*`,
`runs/bench_decode_*.*`), so nothing is lost when the runtime ends.

**Runtime:** any GPU. An L4 is enough and burns far fewer units than an A100; total ~1–1.5 h
(the decode benchmark is most of it). The decode timings depend on the card, so the figure records
which one it ran on.

**Needs on GitHub `main`:** `experiments/eval_position.py`, `experiments/bench_decode.py` and the
frontier `opera_lm/incremental.py` (decoder state that grows as log T).

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/opera_scale'   # same root as OPERA_Scale / OPERA_MAX
RUNS = f'{DRIVE}/runs'
DDATA = f'{DRIVE}/data'
LOCAL = '/content/data'
REPO = '/content/opera-lm'
os.makedirs(LOCAL, exist_ok=True)
os.environ['OPERA_RUNS'] = RUNS

In [ ]:
BRANCH = 'main'
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = (f'https://{tok}@github.com/Merna-Khalid/OPERA-LM.git' if tok
       else 'https://github.com/Merna-Khalid/OPERA-LM.git')
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {url} {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log -1 --oneline
%cd {REPO}
for f in ('experiments/eval_position.py', 'experiments/bench_decode.py'):
    assert os.path.exists(f), f'{f} missing: push it to {BRANCH} first'
assert 'state_numel' in open('opera_lm/incremental.py').read(), \
    'old decoder on this branch: push the frontier OperaDecoder first'
print('repo OK')

In [ ]:
# the Triton kernels must be correct on THIS card before any OPERA number is trusted
!python -m opera_lm.triton_kernel --test

## 2. Which runs exist

Lists every finished OPERA rung (`fw_*`) and byte-transformer rung (`tfb_*`) in `repr_summary.json`.
Tokenized transformers (`tft_*`) are left out of the length figure: their positions are tokens, not
bytes.

In [ ]:
import json, re, sys
sys.path.insert(0, 'experiments')
import ladder
summ = json.load(open(f'{RUNS}/repr_summary.json'))
prev = {}
if os.path.exists(f'{RUNS}/previous/repr_summary.json'):     # read-only: nothing is written
    prev = json.load(open(f'{RUNS}/previous/repr_summary.json'))
    print(f"previous/repr_summary.json: {len(prev)} rows, "
          f"{sum(k not in summ for k in prev)} of them not in the current summary")
# the run FOLDERS are the ground truth; the summary is only an index
folders = sorted(f for f in os.listdir(RUNS) if os.path.isdir(f'{RUNS}/{f}')
                 and re.match(r'^(fw|tfb)_d\d+_L\d+_r\d+(_qknorm)?$', f))
def row(a):
    return summ.get(a) or prev.get(a)
def show(prefix):
    rows = [f for f in folders if f.startswith(prefix)]
    for k in rows:
        r = row(k)
        src = 'summary' if k in summ else 'previous' if k in prev else 'folder only'
        info = (f"{r['params'] / 1e6:7.1f}M  BPB {r['bpb']:.4f}  "
                f"test {os.path.basename(r.get('test_pkl') or '-')}") if r else ''
        print(f"  {k:30s} [{src:11s}] {info}")
    return rows
print('OPERA:'); OPERA_ARMS = show('fw_')
print('byte transformer:'); TFB_ARMS = show('tfb_')

### 2b. Restore the OPERA rows into the summary (one-time, safe to re-run)

The OPERA ladder and 172M rows live in `runs/previous/repr_summary.json` (archived before the QK-norm
transformer re-runs); the current summary started fresh. `ladder.py fit` reads only the current file,
so until the rows are back a fit would overwrite `runs/ladder_fit.json` with the flagship alone.
This copies **only `fw_*` rows that the current summary lacks** (current rows always win; the archived
non-QK-norm transformer rows stay archived), after a timestamped backup, via write-then-rename, and
checks the result. A second run finds nothing to add and changes nothing.

In [ ]:
import shutil, time
cur_p, prev_p = f'{RUNS}/repr_summary.json', f'{RUNS}/previous/repr_summary.json'
if not os.path.exists(prev_p):
    print('no previous/repr_summary.json: nothing to restore')
else:
    before = json.load(open(cur_p))
    old = json.load(open(prev_p))
    add = {k: v for k, v in old.items() if k.startswith('fw_') and k not in before}
    no_folder = [k for k in add if not os.path.isdir(f'{RUNS}/{k}')]
    if no_folder:
        print('note: these rows have no run folder on Drive:', no_folder)
    if not add:
        print('nothing to restore: every archived OPERA row is already in the summary')
    else:
        bak = f'{RUNS}/repr_summary.backup_{time.strftime("%Y%m%d_%H%M%S")}.json'
        shutil.copy(cur_p, bak)
        merged = json.load(open(cur_p))            # re-read right before writing
        merged.update({k: v for k, v in add.items() if k not in merged})
        tmp = cur_p + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(merged, f, indent=2)
        os.replace(tmp, cur_p)
        after = json.load(open(cur_p))
        assert all(k in after for k in before), 'a row went missing -- restore the backup'
        assert all(k in after for k in add), 'a restored row is missing'
        print(f'restored {len(add)} OPERA rows: {sorted(add)}')
        print(f'{len(before)} -> {len(after)} rows; backup: {bak}')
    summ = json.load(open(cur_p))                  # the cells below see the restored index

## 3. Length generalization on FineWeb-Edu

Edit `LENGTH_ARMS` to choose the curves. Defaults: OPERA at d1024 and d2048 (the r20 ladder) plus the
172M run, against the stable QK-norm byte transformers at d768/d1024. The two widest transformer
rungs diverged in training and are left out, as in the paper. Every arm is scored on the **same**
documents (held-out FineWeb documents of at least 4,096 bytes, each cut to 4,096), so every
position averages the same text.

In [ ]:
import shutil
want = ['fw_d1024_L2_r20', 'fw_d2048_L2_r20', 'fw_d3072_L2_r28',
        'tfb_d768_L2_r20_qknorm', 'tfb_d1024_L2_r20_qknorm']
# OPERA runs load from their folder; transformer runs need their summary row
LENGTH_ARMS = [a for a in want if a in folders and (a.startswith('fw_') or a in summ)]
missing = [a for a in want if a not in LENGTH_ARMS]
if missing:
    print('not available (skipped):', missing)

LADDER_TEST = 'fineweb_bytes_T1024_4GB.test.pkl'
def test_of(a):
    r = row(a)
    if r and r.get('test_pkl'):
        return os.path.basename(r['test_pkl'])
    # the r20 ladder and the 172M run all trained on the 4 GB pool (paper §4, §5.3)
    print(f'  {a}: no summary row -> its test set is the ladder pool, {LADDER_TEST}')
    return LADDER_TEST
tests = {a: test_of(a) for a in LENGTH_ARMS}
assert len(set(tests.values())) == 1, f'arms were tested on different pools: {tests}'
TEST = tests[LENGTH_ARMS[0]]
src, dst = f'{DDATA}/{TEST}', f'{LOCAL}/{TEST}'
assert os.path.exists(src), f'{src} missing on Drive (OPERA_Scale §3b caches it)'
if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
    shutil.copy(src, dst)

def label(a):
    r = row(a)
    d, L = map(int, re.match(r'^\w+?_d(\d+)_L(\d+)', a).groups())
    n = (r['params'] if r else ladder.n_params(d, L)) / 1e6
    if a.startswith('fw_'):
        return f'OPERA {n:.0f}M'
    return f"byte transformer {n:.0f}M (NoPE{' + QK-norm' if (r or {}).get('qk_norm') else ''})"
LABELS = [label(a) for a in LENGTH_ARMS]
print('arms :', list(zip(LENGTH_ARMS, LABELS)))
print('test :', dst)

In [ ]:
# MIN_ALPHA: 0 keeps every document. FineWeb-Edu is mostly prose; if the printed document count
# or the curves look dominated by tables/code, rerun with 0.8 (prose filter, recorded in the JSON)
MIN_ALPHA = 0.0
MAX_DOCS = 600          # bounds the run time; the same documents for every arm
arms_s = ' '.join(LENGTH_ARMS)
labels_s = ' '.join(f'"{l}"' for l in LABELS)
!python experiments/eval_position.py --arms {arms_s} --labels {labels_s} \
    --test-pkl {dst} --eval-len 4096 --train-len 1024 --min-alpha {MIN_ALPHA} \
    --max-docs {MAX_DOCS} --bin 128 --batch 2 --device cuda --out {RUNS}/position_curve_fineweb
from IPython.display import Image, display
if os.path.exists(f'{RUNS}/position_curve_fineweb.png'):
    display(Image(f'{RUNS}/position_curve_fineweb.png'))
else:
    print('evaluation failed: see the error above')

**Reading it.** The band ratios printed per arm (1025–2048 and 2049–4096 against 1–1024) are the
numbers for the paper. A ratio below 1.0 alone is not the claim: later positions have more context
and are easier for any model. The claim is the **shape**: OPERA's curve keeps falling or stays flat
past the training length, while a transformer's rises.

## 4. Decoding cost vs context

Random weights at the ladder's widths (state size and step time do not depend on training). Each
model's decoder is checked against its own full forward pass before timing. State is the maximum
over each timing window (OPERA's frontier holds popcount(t) nodes, so a sample at a power of two
would understate it). The transformer counts only the K/V entries actually filled.

In [ ]:
import subprocess
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip().replace(' ', '_')
for d in (2048, 3072):
    out = f'{RUNS}/bench_decode_d{d}_{GPU}'
    !python experiments/bench_decode.py --d {d} --layers 2 --max-ctx 65536 --device cuda --out {out}
    if os.path.exists(out + '.png'):
        display(Image(out + '.png'))
    else:
        print(f'd{d} benchmark failed: see the error above')

In [ ]:
# Optional: the CPU version (what the free Hugging Face Space runs on). Colab's CPU has 2 cores,
# so this is slow (~30 min); the Mac run already gave runs_reprs/bench_decode_d512.png.
RUN_CPU = False
if RUN_CPU:
    !python experiments/bench_decode.py --d 512 --layers 2 --max-ctx 32768 --device cpu --out {RUNS}/bench_decode_d512_colab_cpu
    if os.path.exists(f'{RUNS}/bench_decode_d512_colab_cpu.png'):
        display(Image(f'{RUNS}/bench_decode_d512_colab_cpu.png'))

## 5. Numbers for the paper

In [ ]:
import glob
pc = json.load(open(f'{RUNS}/position_curve_fineweb.json'))
print(f"length: {pc['meta']['n_docs']} documents x {pc['meta']['eval_len']} bytes "
      f"(min-alpha {pc['meta']['min_alpha']})")
for a, r in pc['arms'].items():
    b = r['bands']; base = b['1-1024']
    print(f"  {r['label']:42s} " + '  '.join(f'{k}: {v:.4f} ({v / base:.3f}x)' for k, v in b.items()))
for f in sorted(glob.glob(f'{RUNS}/bench_decode_d*.json')):
    bd = json.load(open(f))
    print(f"\ndecode {os.path.basename(f)}  ({bd['meta']['device']}, d{bd['meta']['d']})")
    rows = {n: {r['ctx']: r for r in v} for n, v in bd['rows'].items()}
    for c in sorted(rows['OPERA']):
        o, t = rows['OPERA'][c], rows['transformer'][c]
        print(f"  ctx {c:6d}: state {o['state_mib']:8.3f} vs {t['state_mib']:9.1f} MiB "
              f"({t['state_mib'] / o['state_mib']:7.0f}x)   {o['ms_per_byte']:6.2f} vs {t['ms_per_byte']:6.2f} ms/byte")